In [4]:
import os, sys
from IPython.display import Markdown, display
repo_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
code_path = os.path.abspath(os.path.join(repo_path, 'src/pneu_abm'))
os.chdir(repo_path)
if code_path not in sys.path:
    sys.path.append(code_path)
display(Markdown(f'**Repository root:** {repo_path}'))

**Repository root:** /

# ODD Protocol for the Pneumococcal ABM

## Table of Contents

1. [Overview](#overview)
2. [Design Concepts](#design-concepts)
3. [Details](#details)
4. [Code-Verified Single Timestep Trace](#single-timestep-trace)


# 1. Overview <a name="overview"></a>

## 1.1 Purpose

The model is an individual-level, age-structured, multi-serotype transmission model with vaccination and disease outcomes. Main `Disease` and simulation classes combines demography with disease transmission: [model/disease/disease.py](../model/disease/disease.py) (line 26), [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 30).

## 1.2 Entities, state variables, and scales

### Entities
- Individual agents are rows of `P.I` (a Polars DataFrame) in `DisPopulation`: [model/population/disease_population.py](../model/population/disease_population.py) (line 18).
- The disease process is managed by a `Disease` object and wrapped by `DiseaseModel` that adds observers: [model/disease/disease.py](../model/disease/disease.py) (line 25), [run_scenarios/disease_model.py](../run_scenarios/disease_model.py) (line 41).

### Agent-level state variables (per individual)
- Demography: `age`, `age_days`, `days_at_death`, `age_group`: [model/population/disease_population.py](../model/population/disease_population.py) (line 45).
- Stochastic individual heterogeneity: `quantile`: [model/population/disease_population.py](../model/population/disease_population.py) (line 50).
- Vaccination struct: `no_of_doses`, `on_time`, `vaccine_type`, `final_vaccine_time`: [model/population/disease_population.py](../model/population/disease_population.py) (line 51).
- Infection history/current carriage: `no_of_strains`, `strain_list`, `endTimes`, `no_past_infections`: [model/population/disease_population.py](../model/population/disease_population.py) (line 59).

### Time scale
- The model runs in discrete ticks; one tick is `365 // t_per_year` days: [model/disease/disease.py](../model/disease/disease.py) (line 57), [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 130).
- Baseline parameters define weekly tile steps by with `t_per_year = 52`: [data/scenario_configs/base_params.py](../data/scenario_configs/base_params.py) (line 61).

### Space/contact structure
- Interaction is age-mixing through a contact matrix (`ContactMatrix.C`) built from file input, transposed on load: [run_scenarios/disease_model.py](../run_scenarios/disease_model.py) (line 87), [model/disease/contact_matrix.py](../model/disease/contact_matrix.py) (line 11).
- Contact age classes are configured from pre-defined age bins: [data/scenario_configs/base_params.py](../data/scenario_configs/base_params.py) (line 88), [model/disease/contact_matrix.py](../model/disease/contact_matrix.py) (line 13).

## 1.3 Process overview and scheduling

At each simulation tick in `DisSimulation._main_loop`, the order is:
1. Convert tick to day: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 130).
2. Update demography (ageing/survival/births/migration): [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 132, 133).
3. Call `Disease.update()`: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 136).

Inside `Disease.update()`, the disease-related sequence is:
1. Vaccination rollout checks: [model/disease/disease.py](../model/disease/disease.py) (line 809).
2. Community FOI + exposure sampling (if FOI exists): [model/disease/disease.py](../model/disease/disease.py) (lines 813, 814).
3. External introductions: [model/disease/disease.py](../model/disease/disease.py) (line 817).
4. Disease outcome generation for new infections: [model/disease/disease.py](../model/disease/disease.py) (line 825).
5. Recovery/clearance updates: [model/disease/disease.py](../model/disease/disease.py) (line 828).
6. Observer recording: [model/disease/disease.py](../model/disease/disease.py) (line 831).

# 2. Design Concepts <a name="design-concepts"></a>

## 2.1 Basic principles

- Transmission is age-structured and serotype-specific. FOI is built from age-group infection fractions and the contact matrix, then transformed as $1-e^{-FOI}$: [model/disease/disease.py](../model/disease/disease.py) (line 644).
- Serotypes have transmission multipliers loaded from `strain_list.csv` and grouped for FOI calculation: [model/disease/disease.py](../model/disease/disease.py) (lines 86, 103).
- Co-infection is constrained by `max_no_coinfections` and susceptibility is reduced for already infected individuals: [model/disease/disease.py](../model/disease/disease.py) (lines 478, 965, 1026).

## 2.2 Emergence

- Serotype prevalence emerges from FOI-driven transmission plus external seeding and competition constraints (`no_of_strains` limit): [model/disease/disease.py](../model/disease/disease.py) (lines 965, 1058, 1417).
- IPD/CAP incidence by age and vaccine strata is an emergent output recorded by observers from generated disease events: [model/disease/disease.py](../model/disease/disease.py) (line 1249), [model/observers/obs_disease_by_age.py](../model/observers/obs_disease_by_age.py) (line 101), [model/observers/obs_disease_by_vaccine.py](../model/observers/obs_disease_by_vaccine.py) (line 124).

## 2.3 Adaptation and objectives

- Agents do not explicitly optimize behavior. Their state changes are rule-driven (vaccination eligibility/schedule, infection/recovery, ageing, death, migration): [model/disease/disease.py](../model/disease/disease.py) (lines 842, 902, 941), [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 178).

## 2.4 Learning and prediction

- No explicit learning/memory update policy is implemented beyond cumulative exposure tracking (`no_past_infections`) and vaccine history fields: [model/population/disease_population.py](../model/population/disease_population.py) (lines 51, 64).
- Vaccine impact is modeled through antibody-mediated probabilistic protection functions rather than adaptive policy learning: [model/disease/disease.py](../model/disease/disease.py) (lines 1168, 1188, 1214).

## 2.5 Sensing and interaction

- Individuals do not directly read neighbors; interaction is mediated through age-contact mixing and group prevalence in FOI: [model/disease/disease.py](../model/disease/disease.py) (lines 669, 696, 727).
- Vaccination interaction with infection occurs via vaccine history and serotype-specific antibody tables joined at exposure/outcome steps: [model/disease/disease.py](../model/disease/disease.py) (lines 1091, 1261).

## 2.6 Stochasticity

- Separate RNG streams are initialized for transmission and vaccination; disease RNG streams are pre-created for clinical replicates: [model/disease/disease.py](../model/disease/disease.py) (lines 42, 43, 49).
- Randomness is included to infection seeding, exposure, strain assignment, vaccine timing, infection durations, and disease outcomes: [model/disease/disease.py](../model/disease/disease.py) (lines 360, 582, 996, 1051, 1327, 1481).

## 2.7 Observation

`DiseaseModel` attaches observers for population, prevalence, vaccination rollout, disease by age, disease by vaccine, vaccines delivered, and prevalence by age: [run_scenarios/disease_model.py](../run_scenarios/disease_model.py) (line 50).

Examples of recorded outputs:
- Population size and age distribution: [model/observers/obs_pop.py](../model/observers/obs_pop.py) (lines 114, 119, 125).
- Overall prevalence and serotype fractions: [model/observers/obs_prevalence.py](../model/observers/obs_prevalence.py) (lines 47, 51, 102).
- Vaccines delivered by type at each day: [model/observers/obs_vacc_delivered.py](../model/observers/obs_vacc_delivered.py) (lines 34, 40, 45).
- Age-specific infections and infected counts: [model/observers/obs_prevalence_by_age.py](../model/observers/obs_prevalence_by_age.py) (lines 37, 64, 70).

# 3. Details <a name="details"></a>

## 3.1 Initialization

- `go_single()` creates output path/file naming based on year span and either loads an existing disease file or creates a new one: [model/disease/run.py](../model/disease/run.py) (lines 38, 41, 43, 56, 71).
- `DisSimulation.create_population()` builds `DisPopulation`, sets RNG/death rates, and generates age-structured population: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 68, 74, 76), [model/disease/disease_utils.py](../model/disease/disease_utils.py) (line 163).
- Initial infection seeding uses age-dependent probability (`2 * infected_population_fraction` for age <= 2, otherwise base value), then samples serotypes from initialization proportions: [model/disease/disease.py](../model/disease/disease.py) (lines 545, 582, 585, 602, 613).

## 3.2 Input data

- Baseline model parameters are in `base_params.py` (demography, transmission, vaccination, seeds, time scale, run horizon): [data/scenario_configs/base_params.py](../data/scenario_configs/base_params.py) (lines 10, 33, 50, 61, 63).
- Strain, vaccine, age-specific protection, disease multipliers, and infection duration tables are loaded during `Disease` initialization: [model/disease/disease.py](../model/disease/disease.py) (lines 67, 107, 112, 483).

## 3.3 Submodels

### 3.3.1 Demography submodel
- Ages advance by `period = 365 // t_per_year`; individuals exceeding `days_at_death` are filtered out: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 192, 197, 208).
- Birth and migration flows are computed each tick from per-tick rates (`birth_rates[t]`, `mig_rates[t]`), with fractional residue accumulation for births: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 222, 226, 236).
- New births and migrants are added through `introduce_births_and_migrations()`, where births are age 0 and migrants are sampled from migration age distribution: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 239), [model/disease/disease_utils.py](../model/disease/disease_utils.py) (lines 164, 177, 188).

### 3.3.2 Vaccination submodel
- Vaccine list rows are parsed (including JSON-like list fields), expanded to full year coverage vectors, and aligned to timestep grid: [model/disease/disease_utils.py](../model/disease/disease_utils.py) (lines 112, 119, 147, 154, 159).
- At each tick, each active vaccine schedule computes current rollout year and applies on-time/late coverage with vaccine-specific targeting function: [model/disease/disease.py](../model/disease/disease.py) (lines 857, 861, 864).
- Late doses are represented through negative `on_time` values that encode future vaccination day and are revisited in later ticks: [model/disease/disease.py](../model/disease/disease.py) (lines 360, 406, 449).

### 3.3.3 Transmission and acquisition submodel
- FOI is built from age-group strain prevalence fractions and contact matrix mixing, with serotype multiplier groups: [model/disease/disease.py](../model/disease/disease.py) (lines 656, 669, 700, 727).
- Probability of exposure per age group is transformed as $1-e^{-FOI}$ and then used in age-group binomial draws: [model/disease/disease.py](../model/disease/disease.py) (lines 667, 996).
- Eligibility excludes individuals already at `max_no_coinfections`; susceptibility is reduced by existing infections via `1 - reduction_in_susceptibility * no_of_strains` clipped at 0: [model/disease/disease.py](../model/disease/disease.py) (lines 965, 1026).
- Exposed strain is sampled from a dynamic strain distribution weighted by transmission multipliers and optional noise term: [model/disease/disease.py](../model/disease/disease.py) (lines 774, 790, 795, 1051).
- If vaccinated, acquisition probability uses antibody waning and a logistic protection expression (`prob_of_transmission`): [model/disease/disease.py](../model/disease/disease.py) (lines 1109, 1168, 1214).

### 3.3.4 Infection duration and recovery submodel
- Infection duration is sampled from an age-specific exponential distribution: [model/disease/disease.py](../model/disease/disease.py) (lines 1476, 1481).
- Carriage end times are appended on infection and expired infections are removed by exploding, filtering `endTimes > day`, regrouping, and updating counts: [model/disease/disease.py](../model/disease/disease.py) (lines 902, 916, 936, 1137).

### 3.3.5 Disease outcome submodel
- New infections are passed to `check_disease()` where vaccine-antibody and age-protection data are joined with age-serotype disease multipliers: [model/disease/disease.py](../model/disease/disease.py) (lines 1249, 1261, 1271).
- Disease probability is computed from log-antibodies; final disease draw uses `prob_of_disease * dis_multiplier`: [model/disease/disease.py](../model/disease/disease.py) (lines 1188, 1319, 1330, 1347).
- Outcomes are split into `ipd` vs `cap` using age-specific `ipd_fraction_by_age_group`: [model/disease/disease.py](../model/disease/disease.py) (lines 491, 1375).
- Clinical-model replicate index is recorded as `clinical_model_no`, and `P.disease_pop` is reset at start-of-year then accumulated within year: [model/disease/disease.py](../model/disease/disease.py) (lines 1325, 1397, 1400, 1405).

### 3.3.6 External exposure submodel
- At intervals defined by `external_exposure_check_period`, random external strains are sampled and assigned to susceptible unvaccinated individuals with probability scaling by `external_exposure_prob`: [model/disease/disease.py](../model/disease/disease.py) (lines 500, 1421, 1431, 1442).
- External infections then go through the same duration assignment and state update pattern as internal transmission: [model/disease/disease.py](../model/disease/disease.py) (lines 1454, 1469).

# 4. Code-Verified Single Timestep Trace <a name="single-timestep-trace"></a>

Given tick `t`, with `day = t * (365 // t_per_year)`, one full iteration does:

1. Demography update (if enabled): agents age, deaths are removed, births/migrants added. Sources: [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (lines 130, 132, 197, 222, 239).
2. Vaccination update: current schedules evaluated, on-time and late doses assigned, agent vaccine struct updated. Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 842, 861, 879).
3. FOI and exposure sampling: age-group exposures drawn, strain sampled, vaccine-mediated acquisition filtering applied. Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 813, 941, 996, 1051, 1118).
4. External introductions (if due): imported strains sampled and applied to eligible hosts. Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 817, 1421).
5. Disease outcomes for newly infected: probability-based CAP/IPD outcomes sampled, recorded into `P.disease_pop`/observer channels. Sources: [model/disease/disease.py](../model/disease/disease.py) (lines 825, 1249, 1375, 1405).
6. Recovery: infections with end time <= day are removed from each agent. Source: [model/disease/disease.py](../model/disease/disease.py) (lines 828, 902).
7. Observer writes: prevalence, disease, vaccine, and population summaries are persisted. Source: [model/disease/disease.py](../model/disease/disease.py) (line 831), [run_scenarios/disease_model.py](../run_scenarios/disease_model.py) (line 50).

This sequence is repeated for all ticks in the configured simulation horizon (`years`, `t_per_year`): [data/scenario_configs/base_params.py](../data/scenario_configs/base_params.py) (lines 61, 63), [model/disease/disease_simulation.py](../model/disease/disease_simulation.py) (line 128).